# Exercise 03 — Induction Heads (write it yourself)

Same task as `01_notebooks/03_induction_heads.ipynb`, but the core logic is
blanked out. Fill in the `# YOUR CODE HERE` sections. Each exercise has a
self-check cell right after it — run it, don't just trust your code by eye.

If you get stuck: expand the `<details>` hint, or as a last resort compare
against [`../01_notebooks/03_induction_heads.ipynb`](../01_notebooks/03_induction_heads.ipynb),
which has the full working version with real output already saved.

Recap: an induction head attends from the current token back to the token
that followed the *previous* occurrence of the current token, effectively
implementing "if I've seen `[A][B]` before and see `[A]` again, predict `[B]`."


In [ ]:
import torch
from transformer_lens import HookedTransformer, utils

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2", device=device)


## Exercise 1 — build the synthetic induction test

Write `induction_batch(model, seq_len=25, batch=4, seed=0)` that returns a
tensor of token ids with shape `[batch, 1 + 2*seq_len]`: one random prefix
token, followed by a random block of `seq_len` tokens, followed by an exact
repeat of that same block.

<details>
<summary>Hint</summary>

`torch.randint(0, model.cfg.d_vocab, (batch, seq_len))` gives you one random
block. Build the prefix the same way with shape `(batch, 1)`. Concatenate
`[prefix, half, half]` along the sequence dimension (`dim=1`), then move the
result to `device`.
</details>


In [ ]:
def induction_batch(model, seq_len=25, batch=4, seed=0):
    torch.manual_seed(seed)
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# self-check
_tokens = induction_batch(model, seq_len=25, batch=4, seed=0)
assert _tokens.shape == (4, 51), f"expected shape (4, 51), got {tuple(_tokens.shape)}"
_seq_len = 25
first_half = _tokens[:, 1:1 + _seq_len]
second_half = _tokens[:, 1 + _seq_len:1 + 2 * _seq_len]
assert torch.equal(first_half, second_half), "second half should be an exact repeat of the first half"
print("Exercise 1 passed")


## Exercise 2 — score every head for "induction-y-ness"

Write `induction_scores(cache, seq_len, n_layers, n_heads)` returning a
`[n_layers, n_heads]` tensor. For each head, average (over batch and
position) the attention probability paid from each destination position in
the *second* half to the source position exactly `seq_len` steps back — that
source is "the token that followed this same token last time."

<details>
<summary>Hint</summary>

For layer `l`, get `pattern = cache[utils.get_act_name("pattern", l)]`,
shape `[batch, head, dest, src]`. `pattern.diagonal(offset=-seq_len + 1,
dim1=-2, dim2=-1)` pulls out exactly the "dest - seq_len" diagonal for every
head and batch element at once — then average it over batch and position
(`dim=(0, -1)`).
</details>


In [ ]:
def induction_scores(cache, seq_len, n_layers, n_heads):
    scores = torch.zeros(n_layers, n_heads)
    for layer in range(n_layers):
        # YOUR CODE HERE
        raise NotImplementedError
    return scores


In [ ]:
# self-check
_tokens = induction_batch(model, seq_len=25, batch=4, seed=0)
_logits, _cache = model.run_with_cache(_tokens)
_scores = induction_scores(_cache, 25, model.cfg.n_layers, model.cfg.n_heads)
assert _scores.shape == (model.cfg.n_layers, model.cfg.n_heads)
assert _scores.max().item() > 0.7, f"expected a clear induction head (score > 0.7), best was {_scores.max().item():.3f}"
_layer, _head = divmod(_scores.flatten().argmax().item(), model.cfg.n_heads)
print(f"Exercise 2 passed — top head: layer {_layer} head {_head}, score {_scores.max().item():.3f}")


## Visualize

Given, not an exercise — confirm your top head visually against
`01_notebooks/03_induction_heads.ipynb`'s saved output.


In [ ]:
import plotly.express as px

tokens = induction_batch(model)
logits, cache = model.run_with_cache(tokens)
scores = induction_scores(cache, 25, model.cfg.n_layers, model.cfg.n_heads)
px.imshow(scores, labels=dict(x="head", y="layer"), title="Induction score by head",
          color_continuous_scale="Blues").show()
